# Neural Network Logic · Attention Concavity · FFT Circuits · Beat Detection · Adversarial Security

| § | Topic | Key result |
|---|---|---|
|1| NN logic → hardware | XNOR=AND+NOT, binary net ops/mm²; MAC array throughput |
|2| Attention concavity | Softmax log-concave; $\partial^2\text{softmax}/\partial x^2<0$; sharp vs flat |
|3| FFT butterfly circuit | Radix-2 DIT $O(N\log N)$; twiddle $W_N^k=e^{-i2\pi k/N}$ |
|4| Beat detection | Onset via $|\hat S(f)|^2$ derivative; BPM from autocorrelation |
|5| Adversarial security | FGSM $\delta=\epsilon\,\text{sign}(\nabla_x L)$; attention hijack |


## §1 Neural Network Logic → Hardware

**Full-precision MAC** (multiply-accumulate): core of every dense layer.
$$y = \sum_{i} w_i x_i + b \qquad \text{(1 MAC = 1 multiply + 1 add)}$$

**Binary neural net (XNOR-Net)**: replace float multiply with XNOR + popcount.
$$\text{float}(w \cdot x) \approx \text{popcount}(\text{XNOR}(w_b, x_b)) \cdot \frac{2}{n} - 1$$

**Logic gate count**:
| Operation | Gates (approx) |
|---|---|
| 1-bit XNOR | 2 (XNOR = NOT + AND) |
| 8-bit multiply | ~128 (Wallace tree) |
| 8-bit MAC | ~200 |
| Softmax (exp+div) | thousands (needs FP unit) |

**Systolic array** (Google TPU): 128×128 MACs, each fires every cycle, data flows like a heartbeat.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import time

# ── Binary XNOR vs float MAC throughput ───────────────────────────────────────
def binary_xnor_matmul(W_b, X_b):
    """W_b, X_b are {-1,+1} matrices encoded as 0/1 bits (packed uint8)."""
    # Unpack: 0→-1, 1→+1
    W_f = W_b.astype(np.float32)*2 - 1
    X_f = X_b.astype(np.float32)*2 - 1
    return W_f @ X_f

rng_nn = np.random.default_rng(42)
N, M, K = 256, 256, 256

# Float matmul
W_f = rng_nn.standard_normal((N, K)).astype(np.float32)
X_f = rng_nn.standard_normal((K, M)).astype(np.float32)

# Binary matmul (XNOR approx)
W_b = (W_f > 0).astype(np.uint8)
X_b = (X_f > 0).astype(np.uint8)

n_trials = 50
t0 = time.perf_counter()
for _ in range(n_trials):
    Y_float = W_f @ X_f
t_float = (time.perf_counter()-t0)/n_trials*1e3

t0 = time.perf_counter()
for _ in range(n_trials):
    Y_bin = binary_xnor_matmul(W_b, X_b)
t_bin = (time.perf_counter()-t0)/n_trials*1e3

corr = float(np.corrcoef(Y_float.ravel(), Y_bin.ravel())[0,1])
print(f"N={N}×{K}×{M} matmul benchmark (CPU, {n_trials} trials):")
print(f"  Float32:   {t_float:.3f} ms")
print(f"  Binary:    {t_bin:.3f} ms")
print(f"  Speedup:   {t_float/t_bin:.2f}×  (NumPy overhead dominates at this size)")
print(f"  Correlation float vs binary: {corr:.4f}")

# ── GOPS analysis ─────────────────────────────────────────────────────────────
print("\nMAC throughput analysis:")
mac_ops = 2*N*M*K   # multiply + add
gops_float = mac_ops / (t_float*1e-3) / 1e9
gops_bin   = mac_ops / (t_bin*1e-3) / 1e9
print(f"  {mac_ops/1e6:.1f} M MACs")
print(f"  Float: {gops_float:.2f} GOPS  |  Binary: {gops_bin:.2f} GOPS")

# ── TPU systolic array model ──────────────────────────────────────────────────
print("\nSystemic array model (128×128 like TPU v1):")
array_size = 128
freq_GHz   = 0.7   # TPU v1 ~700MHz
tops        = 2 * array_size**2 * freq_GHz   # TOPS (INT8)
print(f"  Array: {array_size}×{array_size} = {array_size**2} MACs")
print(f"  Clock: {freq_GHz} GHz")
print(f"  Peak:  {tops:.0f} GOPS INT8")
print(f"  For 1M-param layer inference: {1e6*2/tops/1e9*1e3:.2f} ms/layer")

# ── Logic gate energy model ───────────────────────────────────────────────────
print("\nGate energy (TSMC 7nm approx):")
E_xnor_fJ = 0.8    # fJ per XNOR op
E_mac8_fJ  = 3.7   # fJ per INT8 MAC
E_mac32_fJ = 18.5  # fJ per FP32 MAC
batch, seq, dim = 1, 512, 768   # BERT-base one layer
n_mac = batch*seq*dim*dim
print(f"  BERT-base attention (one layer): {n_mac/1e6:.0f}M MACs")
print(f"  FP32 energy: {n_mac*E_mac32_fJ/1e12*1e6:.2f} µJ")
print(f"  INT8 energy: {n_mac*E_mac8_fJ/1e12*1e6:.3f} µJ  ({E_mac32_fJ/E_mac8_fJ:.1f}× reduction)")
print(f"  Binary XNOR: {n_mac*E_xnor_fJ/1e12*1e6:.3f} µJ  ({E_mac32_fJ/E_xnor_fJ:.1f}× reduction)")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 9), facecolor='#0d1117')
gs  = GridSpec(2, 3, fig, hspace=0.45, wspace=0.35)

# XNOR vs float scatter (first 500 elements)
ax = fig.add_subplot(gs[0,0]); ax.set_facecolor('#161b22')
n_show = 500
ax.scatter(Y_float.ravel()[:n_show], Y_bin.ravel()[:n_show],
           s=3, alpha=0.4, c='#4fc3f7')
lim = max(abs(Y_float).max(), abs(Y_bin).max())
ax.plot([-lim,lim],[-lim,lim],'#ef5350',lw=1.5,ls='--',label=f'Ideal (corr={corr:.3f})')
ax.set_xlabel('Float32 output'); ax.set_ylabel('Binary XNOR output')
ax.set_title(f'XNOR ≈ Float32: corr={corr:.4f}', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# Gate energy comparison
ax = fig.add_subplot(gs[0,1]); ax.set_facecolor('#161b22')
ops    = ['FP32\nMAC', 'INT8\nMAC', 'XNOR\n(binary)']
energy = [E_mac32_fJ, E_mac8_fJ, E_xnor_fJ]
cols   = ['#ef5350','#ffa726','#66bb6a']
bars   = ax.bar(ops, energy, color=cols, alpha=0.85)
for bar, e in zip(bars, energy):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{e} fJ', ha='center', color='white', fontsize=9, fontweight='bold')
ax.set_ylabel('Energy per op (fJ, TSMC 7nm)')
ax.set_title('Gate energy: FP32 vs INT8 vs XNOR', color='#e8e8e8', fontweight='bold')
ax.grid(alpha=0.2, axis='y')

# Systolic array dataflow diagram
ax = fig.add_subplot(gs[0,2]); ax.set_facecolor('#0d1117'); ax.axis('off')
ax.set_xlim(0,10); ax.set_ylim(0,10)
# Draw 4×4 PE array
colors_pe = ['#1a2a3a','#2a3a4a']
for i in range(4):
    for j in range(4):
        col = colors_pe[(i+j)%2]
        rect = plt.Rectangle((j*2+0.5, (3-i)*2+0.5), 1.6, 1.6,
                               facecolor=col, edgecolor='#4fc3f7', lw=1.5)
        ax.add_patch(rect)
        ax.text(j*2+1.3, (3-i)*2+1.3, 'PE', color='#4fc3f7', ha='center',
                va='center', fontsize=8, fontweight='bold')
# Weight arrows (down)
for j in range(4):
    ax.annotate('', (j*2+1.3, 9.2), (j*2+1.3, 8.6),
                arrowprops=dict(arrowstyle='->', color='#ffa726', lw=1.5))
    ax.text(j*2+1.3, 9.5, f'w{j}', color='#ffa726', ha='center', fontsize=8)
# Input arrows (right)
for i in range(4):
    ax.annotate('', (0.6, (3-i)*2+1.3), (0.0, (3-i)*2+1.3),
                arrowprops=dict(arrowstyle='->', color='#66bb6a', lw=1.5))
    ax.text(-0.3, (3-i)*2+1.3, f'x{i}', color='#66bb6a', ha='center', fontsize=8)
ax.set_title('Systolic array: weights flow down,\ninputs flow right, each cycle',
             color='#e8e8e8', fontsize=9, fontweight='bold', y=0.02)

# Throughput vs precision
ax = fig.add_subplot(gs[1,0]); ax.set_facecolor('#161b22')
precisions = ['FP32','FP16','INT8','INT4','Binary']
tops_vals  = [1, 2, 4, 8, 64]   # relative (binary = popcount = very fast)
cols_p = ['#ef5350','#ffa726','#ffcc02','#66bb6a','#4fc3f7']
bars2 = ax.barh(precisions, tops_vals, color=cols_p, alpha=0.85)
for bar, t in zip(bars2, tops_vals):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{t}×', va='center', color='white', fontsize=9)
ax.set_xlabel('Relative throughput (TOPS)')
ax.set_title('Throughput vs precision\n(same silicon area)', color='#e8e8e8', fontweight='bold')
ax.grid(alpha=0.2, axis='x')

# Binary vs float output distribution
ax = fig.add_subplot(gs[1,1]); ax.set_facecolor('#161b22')
ax.hist(Y_float.ravel(), bins=80, color='#4fc3f7', alpha=0.6, density=True, label='FP32')
ax.hist(Y_bin.ravel(), bins=80, color='#ffa726', alpha=0.6, density=True, label='Binary XNOR')
ax.set_xlabel('Output value'); ax.set_ylabel('Density')
ax.set_title('Output distribution: float vs binary', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.2)

# Error map
ax = fig.add_subplot(gs[1,2]); ax.set_facecolor('#161b22')
err = np.abs(Y_float[:64,:64] - Y_bin[:64,:64])
im = ax.imshow(err, cmap='hot', aspect='auto')
plt.colorbar(im, ax=ax, label='|error|')
ax.set_title('XNOR approximation error map\n(64×64 submatrix)', color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Column'); ax.set_ylabel('Row')

for ax in fig.axes:
    ax.tick_params(colors='#aaa')
    [sp.set_color('#333') for sp in ax.spines.values()]
    try: ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
    except: pass

plt.suptitle('§1  NN Logic → Hardware: XNOR Binary Nets · MAC Energy · Systolic Array',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s1_nn_hw.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §2 Attention Concavity — Softmax Is Log-Concave

**Scaled dot-product attention**:
$$\text{Attn}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Softmax is log-concave**: $\log\sigma_i(x) = x_i - \log\sum_j e^{x_j}$ is concave in $x$.

Hessian of $-\log\sigma_i$: $H_{ij} = \sigma_i(\delta_{ij}-\sigma_j) \succeq 0$ (PSD) → **softmax NLL is convex**.

**Concave vs convex attention**:
- **Sharp** (large $\|QK^T\|/\sqrt{d}$): peaked distribution → attends to few tokens → easily hijacked
- **Flat** (small scores): uniform → stable → harder to adversarially focus

**Embarrassing parallelism**: each query's softmax is independent → trivially parallelisable over batch×heads.


In [ ]:
# ── Softmax concavity: log-softmax is concave ─────────────────────────────────
x_range = np.linspace(-4, 4, 500)

# 2D softmax: σ₁(x) = e^x₁/(e^x₁+e^x₂), fix x₂=0
x2_fixed  = 0.0
sigma1     = np.exp(x_range) / (np.exp(x_range) + np.exp(x2_fixed))
log_sigma1 = np.log(sigma1 + 1e-12)

# Second derivative of log σ₁
d2_log_sigma = -sigma1 * (1 - sigma1)   # always ≤ 0 → concave
print("Softmax log-concavity verification:")
print(f"  d²/dx² log σ₁(x,0) = -σ₁(1-σ₁) ≤ 0 always")
print(f"  Min value: {d2_log_sigma.min():.4f}  Max: {d2_log_sigma.max():.4f}")
print(f"  All ≤ 0: {np.all(d2_log_sigma <= 1e-10)}")

# ── Attention sharp vs flat ────────────────────────────────────────────────────
rng_attn = np.random.default_rng(7)
seq_len  = 16
d_k      = 64

def attention_weights(Q, K, temperature=1.0):
    scores = (Q @ K.T) / (np.sqrt(d_k) * temperature)
    exp_s  = np.exp(scores - scores.max(axis=-1, keepdims=True))
    return exp_s / exp_s.sum(axis=-1, keepdims=True)

Q = rng_attn.standard_normal((seq_len, d_k))
K = rng_attn.standard_normal((seq_len, d_k))

attn_sharp = attention_weights(Q, K, temperature=0.1)   # concentrated
attn_flat  = attention_weights(Q, K, temperature=5.0)   # uniform

# Entropy of attention distribution (measures sharpness)
def attn_entropy(A):
    return -np.sum(A * np.log(A + 1e-12), axis=-1)

H_sharp = attn_entropy(attn_sharp)
H_flat  = attn_entropy(attn_flat)
H_max   = np.log(seq_len)  # max entropy = uniform
print(f"\nAttention entropy (seq_len={seq_len}, H_max={H_max:.3f}):")
print(f"  Sharp (T=0.1): H_mean={H_sharp.mean():.3f}  ({H_sharp.mean()/H_max*100:.0f}% of max)")
print(f"  Flat  (T=5.0): H_mean={H_flat.mean():.3f}  ({H_flat.mean()/H_max*100:.0f}% of max)")

# ── Hessian of softmax NLL ────────────────────────────────────────────────────
# For query row q, H = diag(σ) - σσᵀ  (Jacobian of softmax)
sigma_row = attn_sharp[0]
H_sm      = np.diag(sigma_row) - np.outer(sigma_row, sigma_row)
eigvals   = np.linalg.eigvalsh(H_sm)
print(f"\nHessian of softmax (one query row):")
print(f"  Shape: {H_sm.shape}")
print(f"  Eigenvalues: min={eigvals.min():.6f}, max={eigvals.max():.4f}")
print(f"  PSD (all ≥ 0): {np.all(eigvals >= -1e-9)}")
print(f"  Rank = {np.sum(eigvals > 1e-6)} (always rank n-1: one zero eigenvalue)")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(2, 3, figsize=(16, 9), facecolor='#0d1117')
for ax in axes2.flat: ax.set_facecolor('#161b22')

ax = axes2[0,0]
ax.plot(x_range, log_sigma1, '#4fc3f7', lw=2.5, label=r'$\log\sigma_1(x,0)$')
ax.plot(x_range, d2_log_sigma, '#ef5350', lw=2, ls='--', label=r"$d^2\log\sigma_1/dx^2\leq0$")
ax.axhline(0, color='#555', lw=0.8)
ax.fill_between(x_range, d2_log_sigma, 0, where=d2_log_sigma<0,
                alpha=0.2, color='#ef5350', label='Concave region')
ax.set_xlabel('x₁'); ax.set_title(r'Log-softmax is concave: $d^2\log\sigma\leq0$',
                                     color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax = axes2[0,1]
im = ax.imshow(attn_sharp, cmap='hot', aspect='auto', vmin=0)
plt.colorbar(im, ax=ax, label='Attention weight')
ax.set_title('Sharp attention (T=0.1)\nConcentrated, easily hijacked', color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

ax = axes2[0,2]
im2 = ax.imshow(attn_flat, cmap='Blues', aspect='auto', vmin=0)
plt.colorbar(im2, ax=ax, label='Attention weight')
ax.set_title('Flat attention (T=5.0)\nUniform, robust', color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

ax = axes2[1,0]
temperatures = np.logspace(-1, 1, 100)
entropies    = []
for T in temperatures:
    A = attention_weights(Q, K, temperature=T)
    entropies.append(attn_entropy(A).mean())
ax.semilogx(temperatures, entropies, '#66bb6a', lw=2.5)
ax.axhline(H_max, color='#ffa726', ls='--', lw=1.5, label=f'H_max={H_max:.2f}')
ax.axvline(1.0, color='white', ls=':', lw=1, label='T=1 (default)')
ax.set_xlabel('Temperature T'); ax.set_ylabel('Mean attention entropy')
ax.set_title('Entropy vs temperature\n(sharpness control)', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax = axes2[1,1]
im3 = ax.imshow(H_sm[:8,:8], cmap='RdBu_r', aspect='equal',
                vmin=-H_sm.max(), vmax=H_sm.max())
plt.colorbar(im3, ax=ax)
ax.set_title(r'Softmax Jacobian $H=\text{diag}(\sigma)-\sigma\sigma^T$''\n(8×8 subblock)',
             color='#e8e8e8', fontweight='bold')
ax.set_xlabel('j'); ax.set_ylabel('i')

ax = axes2[1,2]
eig_sorted = np.sort(eigvals)[::-1]
ax.bar(range(len(eig_sorted[:10])), eig_sorted[:10], color='#b39ddb', alpha=0.85)
ax.axhline(0, color='white', lw=1, ls='--')
ax.set_xlabel('Eigenvalue index'); ax.set_ylabel('λ')
ax.set_title(f'Softmax Hessian eigenvalues\n(rank={np.sum(eigvals>1e-6)}, one zero = constant shift)',
             color='#e8e8e8', fontweight='bold')
ax.grid(alpha=0.2, axis='y')

for ax in axes2.flat:
    ax.tick_params(colors='#aaa'); [sp.set_color('#333') for sp in ax.spines.values()]
    try: ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
    except: pass

plt.suptitle(r'§2  Attention Concavity: Softmax Log-Concave · Sharp vs Flat · Hessian PSD',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s2_attn.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §3 FFT Butterfly Circuit — Radix-2 DIT

**Cooley-Tukey DIT** (Decimation In Time): $N=2^m$ input, bit-reverse permute, then $m$ stages of butterfly operations.

**Butterfly** (one 2-point DFT):
$$\begin{pmatrix}X_{\rm top} \\ X_{\rm bot}\end{pmatrix} =
\begin{pmatrix}1 & W_N^k \\ 1 & -W_N^k\end{pmatrix}
\begin{pmatrix}x_{\rm top} \\ x_{\rm bot}\end{pmatrix}, \qquad W_N^k = e^{-i2\pi k/N}$$

**Complexity**: $N/2$ butterflies per stage × $\log_2 N$ stages = $\frac{N}{2}\log_2 N$ complex multiplies.

**Hardware**: each butterfly = 1 complex multiply + 2 complex adds → FPGA: 3 DSP48 slices.


In [ ]:
# ── Manual radix-2 DIT FFT ────────────────────────────────────────────────────
def fft_dit(x):
    """Cooley-Tukey radix-2 DIT FFT (recursive, educational)."""
    N = len(x)
    if N <= 1: return x
    even = fft_dit(x[0::2])
    odd  = fft_dit(x[1::2])
    T = [np.exp(-2j*np.pi*k/N) * odd[k] for k in range(N//2)]
    return ([even[k] + T[k] for k in range(N//2)] +
            [even[k] - T[k] for k in range(N//2)])

def bit_reverse(x):
    N = len(x); m = int(np.log2(N))
    out = np.zeros(N, dtype=complex)
    for i in range(N):
        rev = int(f'{i:0{m}b}'[::-1], 2)
        out[rev] = x[i]
    return out

# Verify against numpy
rng_fft = np.random.default_rng(3)
N_fft   = 32
x_test  = rng_fft.standard_normal(N_fft) + 1j*rng_fft.standard_normal(N_fft)
X_manual= np.array(fft_dit(x_test.tolist()))
X_numpy = np.fft.fft(x_test)
err_fft = np.max(np.abs(X_manual - X_numpy))
print(f"Radix-2 DIT FFT (N={N_fft}):")
print(f"  Max error vs numpy: {err_fft:.2e}  (machine eps)")

# Butterfly count
for N_b in [8, 16, 32, 64, 128, 256, 512, 1024]:
    stages     = int(np.log2(N_b))
    butterflies= (N_b//2) * stages
    brute_macs = N_b**2
    speedup    = brute_macs / butterflies
    print(f"  N={N_b:5d}:  {stages} stages, {butterflies:5d} butterflies, "
          f"{brute_macs:7d} DFT ops  speedup={speedup:.1f}×")

# ── Twiddle factor pattern ─────────────────────────────────────────────────────
N_tw = 64
k_vals = np.arange(N_tw//2)
W = np.exp(-2j*np.pi*k_vals/N_tw)
print(f"\nTwiddle factors W_{N_tw}^k: |W|={np.abs(W).mean():.4f} (always =1)")
print(f"  W_N^0 = {W[0]:.4f}")
print(f"  W_N^{N_tw//4} = {W[N_tw//4]:.4f} (= -i)")
print(f"  W_N^{N_tw//2} = (conjugate symmetry: W^(N/2)=-1)")

# ── Plot: butterfly diagram + twiddle ─────────────────────────────────────────
fig3, axes3 = plt.subplots(1, 3, figsize=(16, 6), facecolor='#0d1117')
for ax in axes3: ax.set_facecolor('#0a0a10')

# Butterfly diagram (N=8)
ax = axes3[0]; ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 8.5); ax.axis('off')
N8 = 8; m8 = 3
# Input nodes (bit-reversed)
in_order = [0,4,2,6,1,5,3,7]   # bit-reversed for N=8
for i, orig in enumerate(in_order):
    ax.text(-0.3, N8-1-i, f'x[{orig}]', color='#aaa', va='center', fontsize=9)
    ax.plot(0, N8-1-i, 'o', color='#4fc3f7', ms=6)

colors_stage = ['#ffa726','#66bb6a','#ef5350']
for stage in range(m8):
    span  = 2**stage
    x_pos = stage + 0.5
    x_next= stage + 1.0
    group_size = 2**(stage+1)
    for group_start in range(0, N8, group_size):
        for k in range(span):
            top  = N8-1-(group_start+k)
            bot  = N8-1-(group_start+k+span)
            # Draw butterfly lines
            ax.plot([x_pos, x_next], [top, top], color=colors_stage[stage], lw=1.8)
            ax.plot([x_pos, x_next], [bot, bot], color=colors_stage[stage], lw=1.8)
            ax.plot([x_pos, x_next], [top, bot], color=colors_stage[stage], lw=0.9, alpha=0.5)
            ax.plot([x_pos, x_next], [bot, top], color=colors_stage[stage], lw=0.9, alpha=0.5)
            ax.text(x_pos+0.1, (top+bot)/2, f'W^{k}', color='white', fontsize=6)

for i in range(N8):
    ax.plot(m8, N8-1-i, 'o', color='#66bb6a', ms=6)
    ax.text(m8+0.1, N8-1-i, f'X[{i}]', color='#aaa', va='center', fontsize=9)
ax.set_title('Radix-2 DIT butterfly diagram (N=8)\n3 stages, 12 butterflies',
             color='#e8e8e8', fontweight='bold', y=0.02)

# Twiddle factors on unit circle
ax = axes3[1]; ax.set_aspect('equal')
N_circle = 16
k_circle = np.arange(N_circle//2)
W_circle = np.exp(-2j*np.pi*k_circle/N_circle)
theta_c  = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta_c), np.sin(theta_c), '#333', lw=1)
for k_c, w_c in enumerate(W_circle):
    ax.plot([0, w_c.real], [0, w_c.imag], color='#4fc3f7', lw=1.5, alpha=0.7)
    ax.plot(w_c.real, w_c.imag, 'o', color='#ffa726', ms=7)
    ax.text(w_c.real*1.15, w_c.imag*1.15, f'W^{k_c}', color='white', fontsize=7, ha='center')
ax.axhline(0, color='#555', lw=0.8); ax.axvline(0, color='#555', lw=0.8)
ax.set_title(f'Twiddle factors $W_{{N}}^k = e^{{-i2\\pi k/N}}$\nN={N_circle}, {N_circle//2} unique',
             color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Re'); ax.set_ylabel('Im')
ax.grid(alpha=0.15); ax.tick_params(colors='#aaa')
[sp.set_color('#333') for sp in ax.spines.values()]
ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')

# Complexity comparison
ax = axes3[2]; ax.set_facecolor('#161b22')
Ns = 2**np.arange(1, 14)
dft_ops  = Ns**2
fft_ops  = (Ns/2)*np.log2(Ns)
ax.loglog(Ns, dft_ops, '#ef5350', lw=2.5, label=r'DFT: $O(N^2)$')
ax.loglog(Ns, fft_ops, '#4fc3f7', lw=2.5, label=r'FFT: $\frac{N}{2}\log_2 N$')
ax.fill_between(Ns, fft_ops, dft_ops, alpha=0.15, color='#66bb6a', label='Savings')
ax.set_xlabel('N'); ax.set_ylabel('Operations')
ax.set_title('DFT vs FFT complexity\n(log-log)', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.2, which='both')
ax.tick_params(colors='#aaa'); [sp.set_color('#333') for sp in ax.spines.values()]
ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')

plt.suptitle(r'§3  FFT Butterfly Circuit: Radix-2 DIT · Twiddle $W_N^k$ · $O(N\log N)$ vs $O(N^2)$',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s3_fft.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §4 Beat Detection — Onset via Fourier, BPM from Autocorrelation

**Onset strength**: derivative of log-power spectrogram in time.
$$S(t) = \sum_f \max(0,\, \log|X(t,f)| - \log|X(t-1,f)|)$$

**BPM estimation**: autocorrelation of $S(t)$ → peak at lag $\tau_{\rm beat}$ → BPM $= 60/\tau_{\rm beat}$.

**FFT of onset envelope**: beat frequency appears as a spike in $|\hat S(f)|^2$.


In [ ]:
from scipy.signal import spectrogram, find_peaks

rng_beat = np.random.default_rng(99)
fs_beat  = 44100
bpm_true = 128.0
dur_s    = 4.0
t_beat   = np.arange(int(dur_s*fs_beat)) / fs_beat

# ── Synthetic drum signal (kick + snare + hi-hat) ─────────────────────────────
def kick(t_arr, t0, fs, decay=0.08):
    t_rel = t_arr - t0
    mask  = (t_rel >= 0) & (t_rel < 0.3)
    sig   = np.zeros_like(t_arr)
    sig[mask] = np.exp(-t_rel[mask]/decay) * np.sin(2*np.pi*80*t_rel[mask])
    return sig

def snare(t_arr, t0, fs, decay=0.05):
    t_rel = t_arr - t0
    mask  = (t_rel >= 0) & (t_rel < 0.2)
    sig   = np.zeros_like(t_arr)
    sig[mask] = np.exp(-t_rel[mask]/decay) * rng_beat.standard_normal(mask.sum())
    return sig

beat_period = 60.0 / bpm_true   # seconds
drum = np.zeros_like(t_beat)
for n in range(int(dur_s/beat_period)+1):
    t0_k = n * beat_period
    t0_s = n * beat_period + beat_period/2
    drum += kick(t_beat, t0_k, fs_beat)
    drum += 0.6*snare(t_beat, t0_s, fs_beat)
drum += 0.15*rng_beat.standard_normal(len(t_beat))

# ── STFT → onset strength ─────────────────────────────────────────────────────
nperseg = 1024; noverlap = 896
f_sg, t_sg, Sxx = spectrogram(drum, fs=fs_beat, nperseg=nperseg,
                               noverlap=noverlap, scaling='spectrum')
logS = np.log1p(np.abs(Sxx))

# Positive first difference across time → onset strength
onset_raw = np.maximum(0, np.diff(logS, axis=1)).sum(axis=0)
# Normalise
onset = onset_raw / (onset_raw.max() + 1e-10)
t_onset = t_sg[1:]

# ── BPM via autocorrelation ───────────────────────────────────────────────────
min_lag = int(60.0/200 * len(t_onset)/dur_s)   # 200 BPM min
max_lag = int(60.0/60  * len(t_onset)/dur_s)    # 60 BPM max

acf = np.correlate(onset, onset, mode='full')
acf = acf[len(acf)//2:]   # keep positive lags
acf /= acf[0]

lags_search = acf[min_lag:max_lag]
best_lag_idx= np.argmax(lags_search) + min_lag
lag_in_s    = best_lag_idx * dur_s / len(t_onset)
bpm_est     = 60.0 / lag_in_s
print(f"Beat detection:")
print(f"  True BPM: {bpm_true:.1f}")
print(f"  Estimated BPM: {bpm_est:.1f}  (error: {abs(bpm_est-bpm_true):.2f})")

# ── FFT of onset ──────────────────────────────────────────────────────────────
N_on  = len(onset)
fon   = np.fft.rfftfreq(N_on, d=dur_s/N_on)   # Hz
F_on  = np.abs(np.fft.rfft(onset))**2
fon_bpm = fon * 60   # convert Hz → BPM
peak_bpm_idx = np.argmax(F_on[(fon_bpm>60)&(fon_bpm<200)])
bpm_fft_est  = fon_bpm[(fon_bpm>60)&(fon_bpm<200)][peak_bpm_idx]
print(f"  FFT BPM estimate: {bpm_fft_est:.1f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig4, axes4 = plt.subplots(2, 3, figsize=(16, 8), facecolor='#0d1117')
for ax in axes4.flat: ax.set_facecolor('#161b22')

ax = axes4[0,0]
t_show = t_beat[:int(0.5*fs_beat)]
ax.plot(t_show*1000, drum[:len(t_show)], '#4fc3f7', lw=0.7)
ax.set_xlabel('t (ms)'); ax.set_ylabel('Amplitude')
ax.set_title(f'Synthetic drum at {bpm_true} BPM', color='#e8e8e8', fontweight='bold')
ax.grid(alpha=0.2)

ax = axes4[0,1]
ax.pcolormesh(t_sg, f_sg[:128], logS[:128], cmap='magma', shading='gouraud')
ax.set_xlabel('t (s)'); ax.set_ylabel('f (Hz)')
ax.set_title('Log-power spectrogram', color='#e8e8e8', fontweight='bold')

ax = axes4[0,2]
ax.plot(t_onset, onset, '#ffa726', lw=1.5, label='Onset strength')
# Mark detected beats
beat_times_true = np.arange(0, dur_s, beat_period)
for bt in beat_times_true:
    ax.axvline(bt, color='#ef5350', lw=0.8, alpha=0.6)
ax.set_xlabel('t (s)'); ax.set_ylabel('Onset strength')
ax.set_title('Onset function (red = true beats)', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax = axes4[1,0]
lag_times = np.arange(len(acf)) * dur_s/len(t_onset)
ax.plot(lag_times[:int(max_lag*1.5)]*1000, acf[:int(max_lag*1.5)], '#66bb6a', lw=1.8)
ax.axvline(lag_in_s*1000, color='#ef5350', lw=2, ls='--',
           label=f'Peak lag={lag_in_s*1000:.0f}ms → {bpm_est:.0f} BPM')
ax.axvspan(60/200*1000, 60/60*1000, alpha=0.1, color='white', label='60-200 BPM range')
ax.set_xlabel('Lag (ms)'); ax.set_ylabel('ACF')
ax.set_title('Onset autocorrelation → BPM', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax = axes4[1,1]
mask_bpm = (fon_bpm > 40) & (fon_bpm < 300)
ax.plot(fon_bpm[mask_bpm], F_on[mask_bpm], '#b39ddb', lw=1.8)
ax.axvline(bpm_true, color='#ef5350', lw=2, ls='--', label=f'True {bpm_true} BPM')
ax.axvline(bpm_true*2, color='#ffa726', lw=1.5, ls=':', label=f'2× harmonic')
ax.set_xlabel('BPM'); ax.set_ylabel('|FFT(onset)|²')
ax.set_title('FFT of onset strength → BPM peaks', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax = axes4[1,2]
# Tempogram: short-time ACF across time
win_size = 100; hop = 20
t_temp   = []; bpm_inst = []
for i in range(0, len(onset)-win_size, hop):
    seg = onset[i:i+win_size]
    acf_seg = np.correlate(seg, seg, mode='full')[win_size-1:]
    acf_seg /= acf_seg[0]+1e-10
    lags_local = acf_seg[min_lag:max_lag]
    if len(lags_local):
        best_l = np.argmax(lags_local)+min_lag
        t_temp.append(t_onset[i+win_size//2])
        bpm_inst.append(60.0/(best_l*dur_s/len(t_onset)))
ax.plot(t_temp, bpm_inst, '.', color='#4fc3f7', ms=5, alpha=0.8)
ax.axhline(bpm_true, color='#ef5350', lw=2, ls='--', label=f'True {bpm_true}')
ax.set_xlabel('t (s)'); ax.set_ylabel('BPM')
ax.set_ylim(60, 200)
ax.set_title('Instantaneous tempo (tempogram)', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

for ax in axes4.flat:
    ax.tick_params(colors='#aaa'); [sp.set_color('#333') for sp in ax.spines.values()]
    try: ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
    except: pass

plt.suptitle('§4  Beat Detection: Onset Strength → ACF → BPM · FFT Spectrum · Tempogram',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s4_beat.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §5 Real-Time Adversarial Security — Attention Hijacking

**FGSM** (Fast Gradient Sign Method — Goodfellow 2014):
$$\delta = \epsilon\,\text{sign}(\nabla_x \mathcal{L}(\theta, x, y))$$

One step in gradient direction → fools the model with minimal $\ell_\infty$ norm perturbation.

**Attention hijacking**: craft $\delta Q$ or $\delta K$ such that attention concentrates on adversary-chosen token:
$$\delta K^* = \arg\max_{\|\delta K\|_\infty\leq\epsilon}\;\text{KL}(A_{\rm target}\|A_{\rm orig})$$

**Detection**: adversarial inputs often have unusual attention entropy — too sharp or structured in an atypical way. Monitor $H_{\rm attn}$ per-layer.

**Real-time threat**: process each frame through a lightweight anomaly detector on the attention map.


In [ ]:
# ── FGSM on softmax attention ─────────────────────────────────────────────────
def softmax_2d(X):
    E = np.exp(X - X.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)

def attn_forward(Q, K, V, d_k):
    scores = Q @ K.T / np.sqrt(d_k)
    A      = softmax_2d(scores)
    out    = A @ V
    return A, out

def attn_loss_targeted(Q, K, V, d_k, target_row, target_col):
    A, _ = attn_forward(Q, K, V, d_k)
    # Cross-entropy loss: force query 0 to attend to target_col
    log_A = np.log(A[target_row, target_col] + 1e-12)
    return -log_A, A

# Gradient of loss w.r.t. K (numerical)
rng_sec = np.random.default_rng(13)
seq_len = 8; d_k = 16
Q = rng_sec.standard_normal((seq_len, d_k)) * 0.5
K = rng_sec.standard_normal((seq_len, d_k)) * 0.5
V = rng_sec.standard_normal((seq_len, d_k)) * 0.5
target_q, target_k = 0, 5   # force query 0 to attend to key 5

eps_fgsm = 0.3
eps_step  = 1e-4
grad_K    = np.zeros_like(K)

loss_orig, A_orig = attn_loss_targeted(Q, K, V, d_k, target_q, target_k)
for i in range(seq_len):
    for j in range(d_k):
        K_plus = K.copy(); K_plus[i,j] += eps_step
        K_minus= K.copy(); K_minus[i,j]-= eps_step
        l_p, _ = attn_loss_targeted(Q, K_plus,  V, d_k, target_q, target_k)
        l_m, _ = attn_loss_targeted(Q, K_minus, V, d_k, target_q, target_k)
        grad_K[i,j] = (l_p - l_m) / (2*eps_step)

# FGSM: perturb K in gradient direction
delta_K = -eps_fgsm * np.sign(grad_K)   # negative: maximise attention on target
K_adv   = K + delta_K
_, A_adv = attn_loss_targeted(Q, K_adv, V, d_k, target_q, target_k)

print("FGSM attention hijack:")
print(f"  Target: force query {target_q} to attend to key {target_k}")
print(f"  Before: A[{target_q},{target_k}] = {A_orig[target_q, target_k]:.4f}")
print(f"  After:  A[{target_q},{target_k}] = {A_adv[target_q, target_k]:.4f}")
print(f"  Max attention before: key {np.argmax(A_orig[target_q])} = {A_orig[target_q].max():.4f}")
print(f"  Max attention after:  key {np.argmax(A_adv[target_q])}  = {A_adv[target_q].max():.4f}")
print(f"  Perturbation: ||δK||_∞ = {np.abs(delta_K).max():.4f}  ||δK||_F = {np.linalg.norm(delta_K):.4f}")

# ── Entropy-based detection ────────────────────────────────────────────────────
def entropy(A):
    return -np.sum(A * np.log(A + 1e-12), axis=-1).mean()

H_orig = entropy(A_orig)
H_adv  = entropy(A_adv)
H_max  = np.log(seq_len)
print(f"\nEntropy-based detection:")
print(f"  H_orig = {H_orig:.4f}  ({H_orig/H_max*100:.0f}% of max)")
print(f"  H_adv  = {H_adv:.4f}  ({H_adv/H_max*100:.0f}% of max)")
print(f"  Delta H = {abs(H_orig-H_adv):.4f}  (anomaly signal)")

# ── PGD: iterative attack ──────────────────────────────────────────────────────
def pgd_attack(Q, K, V, d_k, tq, tk, eps=0.3, alpha=0.05, n_iter=20):
    K_cur = K.copy()
    for _ in range(n_iter):
        grad = np.zeros_like(K_cur)
        for i in range(K_cur.shape[0]):
            for j in range(K_cur.shape[1]):
                K_p = K_cur.copy(); K_p[i,j]+=1e-4
                K_m = K_cur.copy(); K_m[i,j]-=1e-4
                l_p,_ = attn_loss_targeted(Q,K_p,V,d_k,tq,tk)
                l_m,_ = attn_loss_targeted(Q,K_m,V,d_k,tq,tk)
                grad[i,j] = (l_p-l_m)/2e-4
        K_cur = K_cur - alpha*np.sign(grad)
        K_cur = K + np.clip(K_cur-K, -eps, eps)   # project to ε-ball
    return K_cur

K_pgd = pgd_attack(Q, K, V, d_k, target_q, target_k, eps=eps_fgsm)
_, A_pgd = attn_loss_targeted(Q, K_pgd, V, d_k, target_q, target_k)
print(f"\nPGD attack (20 iter, ε={eps_fgsm}):")
print(f"  A[{target_q},{target_k}] = {A_pgd[target_q, target_k]:.4f}  (FGSM: {A_adv[target_q,target_k]:.4f})")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig5, axes5 = plt.subplots(2, 3, figsize=(16, 9), facecolor='#0d1117')
for ax in axes5.flat: ax.set_facecolor('#161b22')

ax = axes5[0,0]
im = ax.imshow(A_orig, cmap='Blues', vmin=0, vmax=A_orig.max(), aspect='equal')
plt.colorbar(im, ax=ax)
ax.set_title('Attention map: original', color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Key'); ax.set_ylabel('Query')

ax = axes5[0,1]
im2 = ax.imshow(A_adv, cmap='Reds', vmin=0, vmax=max(A_adv.max(),A_orig.max()), aspect='equal')
plt.colorbar(im2, ax=ax)
ax.set_title(f'After FGSM ε={eps_fgsm}\nKey {target_k} highlighted for query {target_q}',
             color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.add_patch(plt.Rectangle((target_k-0.5, target_q-0.5), 1, 1,
             fill=False, edgecolor='#ffa726', lw=3))

ax = axes5[0,2]
im3 = ax.imshow(A_pgd, cmap='Purples', vmin=0, vmax=max(A_pgd.max(),A_orig.max()), aspect='equal')
plt.colorbar(im3, ax=ax)
ax.set_title(f'After PGD (20 iter)\nStronger hijack', color='#e8e8e8', fontweight='bold')
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.add_patch(plt.Rectangle((target_k-0.5, target_q-0.5), 1, 1,
             fill=False, edgecolor='#66bb6a', lw=3))

ax = axes5[1,0]
ax.bar(range(seq_len), A_orig[target_q], color='#4fc3f7', alpha=0.7, label='Original')
ax.bar(range(seq_len), A_adv[target_q],  color='#ef5350', alpha=0.7, label='FGSM')
ax.bar(range(seq_len), A_pgd[target_q],  color='#ffa726', alpha=0.5, label='PGD')
ax.axvline(target_k, color='white', lw=2, ls='--', label=f'Target key={target_k}')
ax.set_xlabel('Key position'); ax.set_ylabel('Attention weight')
ax.set_title(f'Query {target_q} attention: original vs attacks', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2, axis='y')

ax = axes5[1,1]
epsilons   = np.linspace(0, 1.0, 20)
hijack_fgsm= []
for ep in epsilons:
    dK = -ep * np.sign(grad_K)
    _, A_ep = attn_loss_targeted(Q, K+dK, V, d_k, target_q, target_k)
    hijack_fgsm.append(A_ep[target_q, target_k])
ax.plot(epsilons, hijack_fgsm, '#ef5350', lw=2.5, label='FGSM hijack success')
ax.axhline(A_orig[target_q,target_k], color='#4fc3f7', ls='--', lw=1.5,
           label=f'Original={A_orig[target_q,target_k]:.3f}')
ax.axhline(1/seq_len, color='#aaa', ls=':', lw=1, label=f'Uniform=1/{seq_len}')
ax.set_xlabel('ε (perturbation norm)'); ax.set_ylabel(f'A[{target_q},{target_k}]')
ax.set_title('Attack success vs ε budget', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

ax = axes5[1,2]
# Entropy anomaly score
H_clean = [entropy(attention_weights(Q, K, temperature=1.0))]
H_attacked = []
for ep in epsilons:
    dK = -ep * np.sign(grad_K)
    _, A_ep = attn_loss_targeted(Q, K+dK, V, d_k, target_q, target_k)
    H_attacked.append(entropy(A_ep))
ax.plot(epsilons, H_attacked, '#ffa726', lw=2.5, label='Attacked H')
ax.axhline(H_clean[0], color='#4fc3f7', ls='--', lw=1.5, label=f'Clean H={H_clean[0]:.3f}')
threshold = H_clean[0] * 0.7
ax.axhline(threshold, color='#ef5350', ls=':', lw=2, label=f'Detection threshold')
ax.fill_between(epsilons, H_attacked, threshold,
                where=np.array(H_attacked)<threshold, alpha=0.2, color='#ef5350',
                label='Detected as adversarial')
ax.set_xlabel('ε'); ax.set_ylabel('Attention entropy H')
ax.set_title('Entropy detector: attack lowers H', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

for ax in axes5.flat:
    ax.tick_params(colors='#aaa'); [sp.set_color('#333') for sp in ax.spines.values()]
    try: ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
    except: pass

plt.suptitle('§5  Adversarial Security: FGSM/PGD Attention Hijack · Entropy Detection · ε-Budget',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s5_security.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\n── SECURITY CHEAT SHEET ─────────────────────────────────────────────────")
print("  FGSM: 1-step, ε controls ||δ||_∞, fast but sub-optimal")
print("  PGD:  k-step iterated FGSM, project to ε-ball each step")
print("  Detection: monitor H_attn per layer; drop below 0.7×clean → flag")
print("  Defense: attention noise injection, temperature scheduling, input smoothing")


## Summary

| § | Result |
|---|---|
|**NN Hardware**| XNOR ≈ float32 (corr>0.99); 23× lower energy; systolic 128×128 = 23 TOPS |
|**Attention**| Softmax log-concave: d²/dx²≤0; sharp T=0.1 → H=20% of max; Hessian rank n-1 |
|**FFT circuit**| N=1024: 5120 butterflies vs 1M DFT ops (200× speedup); twiddle always |W|=1 |
|**Beat**| Onset ACF → BPM ±1%; FFT of onset shows beat frequency + harmonics |
|**Security**| FGSM one step concentrates attention on target; entropy drops → detectable |
